In [3]:
# =======================================================================
# v13 — E4 asymmetric shape test — k_in=96, k_out=64
# =======================================================================
#
# Goal:
# Test whether the "same-shape" condition is truly required as a bare minimum
# by intentionally setting k_in > k_out while keeping the rest identical.
#
# Configuration:
#   input embedding rank   k_in  = 96
#   output head rank       k_out = 64
#
# This keeps a strong output rank while expanding input embedding capacity.
# Warm-start is adapted for rectangular shapes by copying the first k_out rows
# of B.T into out_to_k.
#
# Predictions (from the theory, not from search):
#   params ≈ 8.40M  (vs 10.0M for k=64, 16% smaller)
#   ep1 test PPL ≈ 60 ± 4
#   ep2 test PPL ≈ 54 ± 3   <-- this must beat the v9 56.66 ceiling to confirm
#   A_in norm growth similar to E4 k=64 (+5% over 8000 steps)
#   CE at step 1000 ≈ 8.9-9.1 nats (comparable to E4 k=64's 8.975)
#
# If PPL_ep2 > 56: condition 3 needs strict inequality, not equality; scaling
#   rule becomes k > d_head, not k >= d_head.
# If PPL_ep2 <= 56: 4-condition theory confirmed. At 7B (d_head=128) the
#   minimum k is 128, giving 32x reduction of the embedding/head system.
#
# Nothing else changes from the k=64 notebook beyond K_IN/K_OUT and RUN_NAME.
# =======================================================================

import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import gc
import math
import shutil
import time
from pathlib import Path
from types import SimpleNamespace

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import GPT2TokenizerFast
from datasets import load_dataset


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    USE_BF16 = torch.cuda.is_bf16_supported()
else:
    USE_BF16 = False


# ── Hyperparameters (identical to E3) ─────────────────────────────────────────
D_MODEL   = 192
N_HEADS   = 4
D_FF_GATE = 512
N_LAYERS  = 8
K_IN      = 96
K_OUT     = 64
DROPOUT   = 0.1
ROPE_BASE = 10000.0
N_POSITIONS = 1024
VOCAB_SIZE  = 50257

BATCH_SIZE   = 8
ACCUM_STEPS  = 8
SEQ_LEN      = 1024
LR           = 3e-4
EPOCHS       = 2
WARMUP_STEPS = 300
GRAD_CLIP    = 1.0

SCHEDULE_HORIZON_EPOCHS = 10
NUM_WORKERS      = 2
VAL_MAX_BATCHES  = None
EVAL_STRIDE      = 512

DRIVE_CACHE_DIR = Path('/content/drive/MyDrive/llm_token_cache/wikitext103_gpt2')
LOCAL_CACHE_DIR = Path('/content/token_cache/wikitext103_gpt2')
TOKENIZER_NAME  = 'gpt2'
TOKENIZE_BATCH_SIZE = 2000

RUN_NAME = 'v13_E4_kin96_kout64'

# ── E4 Diagnosis ─────────────────────────────────────────────────────────────
#
#  THE DISEASE: A is still tied in E3. With k=24, V/k = 2094.
#
#  For any A[v]:
#    OUTPUT path  → touches all 50,257 rows every step, magnitude κ ≈ 17
#    INPUT  path  → touches ~8,192 rows/step (16.3%), magnitude ≈ 0.35
#    AdamW effective update: INPUT contributes ~2% of OUTPUT
#    → A is 97.9% shaped by retrieval geometry, 2.1% by representation quality
#    → 84% of tokens (never in batch) have A[v] chosen purely for retrieval,
#      not for being a useful transformer input
#
#  WHY D/k = 8× understates it:
#    Standard tied head (V×D): output gradient spread over D dims, input path
#    can live in the orthogonal complement. V/D ≈ 262.
#    Factored tied head (V×k): output gradient spread over only k dims. No
#    orthogonal complement to hide in. V/k ≈ 2094. AdamW rounds input path to
#    near-zero in the shared second moment. Absolute harm, not relative.
#
#  THE FIX (E4):
#    Split A → A_in (input path) + A_out (output path). No shared matrix.
#    INPUT:  ids → A_in → B → transformer → h
#    OUTPUT: h → out_to_k → z → normalize(A_out) → logits
#
#  GRADIENT ROUTING (complete):
#    A_in.weight   ← g_in  only  (sparse, batch tokens, free to specialize)
#    A_out.weight  ← g_out only  (dense, all V, free to optimize retrieval)
#    B.weight      ← g_in  only  (unchanged from E2/E3)
#    out_to_k      ← g_out only  (unchanged from E2/E3)
#
#  WARM-START (free fix):
#    Random init: out_to_k and B.T span subspaces with k²/D ≈ 0.5% overlap.
#    First ~1000 steps spent rotating out_to_k to find B's subspace (visible
#    in E3 κ trajectory: slow growth phase 200→1200 before κ plateaus).
#    Fix: warm-start out_to_k from B.T rows to maximize initial overlap.
#    With k_in != k_out, copy B.T[:k_out] into out_to_k.weight.
#
#  PARAMS:
#    E3: A shared = V×k = 1,206,168
#    E4: A_in + A_out = 2×V×k = 2,412,336 (+1,206,168)
#    E4 total ≈ 5,964,528 params (+25.3% vs E3)
#
#  PREDICTIONS:
#    epoch 1 val_recall@1  ≥ 0.34   (vs E3 epoch 1: 0.3169)
#    epoch 2 val_recall@1  ≥ 0.38   (vs E3 epoch 2 est: 0.333)
#    epoch 2 val_ppl       ≤ 65     (vs E3 epoch 2 est: ~75)
#    epoch 1 train_ce drop ≥ 1.3×E3 (decoupled A_in unlocks better h)
#
#  DIAGNOSTIC to log:
#    ‖A_in[v]‖ vs ‖A_out[v]‖ histograms  → do they diverge?
#    cos(A_in[v], A_out[v]) per token     → if → 1: tying was benign
#                                           if → 0: matrices wanted different geometry
#
#  FALSIFICATION:
#    recall < 0.32 at epoch 1 → A-conflict was not the bottleneck
#    ppl > 72 at epoch 2      → gains don't translate to PPL, need per-token scale
#    cos(A_in, A_out) → 1     → E4 rediscovers E3, split was unnecessary
#
# ─────────────────────────────────────────────────────────────────────────────


# ── Dataset ───────────────────────────────────────────────────────────────────
class TokenDataset(Dataset):
    def __init__(self, tokens, seq_len):
        self.tokens  = tokens
        self.seq_len = seq_len
        self.n       = (len(tokens) - 1) // seq_len

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        s = i * self.seq_len
        return self.tokens[s:s + self.seq_len], self.tokens[s + 1:s + self.seq_len + 1]


# ── Building blocks (unchanged) ───────────────────────────────────────────────
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps    = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        orig_dtype = x.dtype
        x32 = x.to(torch.float32)
        rms = x32.pow(2).mean(dim=-1, keepdim=True).add(self.eps).rsqrt()
        return (x32 * rms).to(orig_dtype) * self.weight


class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len, base=10000.0):
        super().__init__()
        assert dim % 2 == 0
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        t        = torch.arange(max_seq_len, dtype=torch.float)
        freqs    = torch.einsum('i,j->ij', t, inv_freq)
        self.register_buffer('cos_cached', freqs.cos()[None, None, :, :], persistent=False)
        self.register_buffer('sin_cached', freqs.sin()[None, None, :, :], persistent=False)

    def apply(self, x, offset=0):
        t   = x.size(-2)
        cos = self.cos_cached[:, :, offset:offset + t, :].to(dtype=x.dtype, device=x.device)
        sin = self.sin_cached[:, :, offset:offset + t, :].to(dtype=x.dtype, device=x.device)
        x_even, x_odd = x[..., ::2], x[..., 1::2]
        out_even = x_even * cos - x_odd * sin
        out_odd  = x_even * sin + x_odd * cos
        return torch.stack((out_even, out_odd), dim=-1).flatten(-2)


# ── E4: FactorizedEmbeddingE4 — THE core change ───────────────────────────────
class FactorizedEmbeddingE4(nn.Module):
    """
    Full path decoupling: A_in ≠ A_out. No tied weights anywhere.

    E3 had a shared A ∈ R^{V×k}. With k=24, V/k=2094, making the output-path
    gradient (dense, magnitude ~17) completely dominate the input-path gradient
    (sparse 16.3% coverage, magnitude ~0.35). AdamW's adaptive denominator
    rounds the input-path contribution to ~2% of the effective update.

    A_in  — optimized to be good transformer inputs (sparse gradient, low mag)
    A_out — optimized for retrieval geometry on S^{k-1} (dense gradient, high mag)
    Both matrices now get 100% of their respective gradient signals.

    Parameter count vs E3: +V×k = +1,206,168 params (+25.3%)
    """
    def __init__(self, vocab_size, d_model, k_in, k_out):
        super().__init__()
        self.A_in  = nn.Embedding(vocab_size, k_in)       # INPUT path only
        self.A_out = nn.Embedding(vocab_size, k_out)      # OUTPUT path only
        self.B     = nn.Linear(k_in, d_model, bias=False) # input upscale
        nn.init.normal_(self.A_in.weight,  0.0, 0.02)
        nn.init.normal_(self.A_out.weight, 0.0, 0.02)
        nn.init.orthogonal_(self.B.weight)

    def embed(self, ids):
        """INPUT path: ids → A_in → B → R^D. Gradient flows to A_in, B only."""
        return self.B(self.A_in(ids))

    def normalized_A_out(self):
        """OUTPUT path: A_out on unit sphere. Gradient flows to A_out only."""
        return F.normalize(self.A_out.weight.float(), dim=-1)   # [V, k]

    def decode_scores(self, z):
        """
        vMF scoring for inference/sampling: logit_v = κ(h) · cos(ẑ, Ã_out_v)
        z: [B, T, k_out] unnormalized from out_to_k
        """
        A_n   = self.normalized_A_out()
        z_n   = F.normalize(z.float(), dim=-1)
        kappa = z.float().norm(dim=-1, keepdim=True)
        return kappa * (z_n @ A_n.T)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff_gate, dropout, n_layers, rope):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads   = n_heads
        self.d_head    = d_model // n_heads
        self.dropout_p = dropout
        self.rope      = rope

        self.norm_attn = RMSNorm(d_model)
        self.norm_ffn  = RMSNorm(d_model)
        self.qkv_proj  = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj  = nn.Linear(d_model, d_model, bias=False)
        self.q_norm    = RMSNorm(self.d_head)
        self.k_norm    = RMSNorm(self.d_head)
        self.w_gate    = nn.Linear(d_model, d_ff_gate, bias=False)
        self.w_up      = nn.Linear(d_model, d_ff_gate, bias=False)
        self.w_down    = nn.Linear(d_ff_gate, d_model, bias=False)
        self.dropout   = nn.Dropout(dropout)

        scale = 1.0 / math.sqrt(2 * n_layers)
        nn.init.normal_(self.qkv_proj.weight, 0.0, 0.02)
        nn.init.normal_(self.out_proj.weight, 0.0, 0.02 * scale)
        nn.init.normal_(self.w_gate.weight,   0.0, 0.02)
        nn.init.normal_(self.w_up.weight,     0.0, 0.02)
        nn.init.normal_(self.w_down.weight,   0.0, 0.02 * scale)

    def attention(self, x):
        b, t, d = x.shape
        q, k, v = self.qkv_proj(x).chunk(3, dim=-1)
        def reshape(z): return z.view(b, t, self.n_heads, self.d_head).transpose(1, 2)
        q, k, v = reshape(q), reshape(k), reshape(v)
        q = self.rope.apply(self.q_norm(q))
        k = self.rope.apply(self.k_norm(k))
        out = F.scaled_dot_product_attention(
            q, k, v, is_causal=True,
            dropout_p=self.dropout_p if self.training else 0.0,
        )
        return self.out_proj(out.transpose(1, 2).contiguous().view(b, t, d))

    def ffn(self, x):
        return self.w_down(self.dropout(F.silu(self.w_gate(x)) * self.w_up(x)))

    def forward(self, x):
        x = x + self.dropout(self.attention(self.norm_attn(x)))
        x = x + self.dropout(self.ffn(self.norm_ffn(x)))
        return x


# ── E4: Main model ─────────────────────────────────────────────────────────────
class BreakthroughMicroTransformerE4(nn.Module):
    """
    E4: Full gradient path decoupling — no shared matrix between input and output.

    All five ingredients now active:

    Ingredient               | Baseline | E2  | E3  | E4
    ─────────────────────────┼──────────┼─────┼─────┼────
    log Z(h) partition fn    |   ✓      |  ✓  |  ✓  |  ✓
    Decoupled B (k→D)        |   ✗      |  ✓  |  ✓  |  ✓
    ‖A_out_v‖ = 1 (sphere)   |   ✗      |  ✓  |  ✓  |  ✓
    Adaptive κ(h) = ‖z‖      |   ✗      |  ✗  |  ✓  |  ✓
    Decoupled A (V×k)        |   ✗      |  ✗  |  ✗  |  ✓  ← E4 addition

    Core disease fixed by E4:
      Shared A with V/k=2094 meant output-path gradient (κ≈17, dense) drowned
      input-path gradient (mag≈0.35, 16.3% sparse) through AdamW's adaptive
      denominator. A was 97.9% a retrieval matrix, 2.1% a representation matrix.
      A_in and A_out can now specialize independently.

    Warm-start:
      out_to_k.weight ← B.weight.T at init.
      Gives 100% subspace alignment vs 0.5% random.
      Eliminates the ~1000-step subspace-rotation tax visible in E3 κ trajectory.
    """
    def __init__(self):
        super().__init__()
        self.embed      = FactorizedEmbeddingE4(VOCAB_SIZE, D_MODEL, K_IN, K_OUT)
        self.rope       = RotaryEmbedding(D_MODEL // N_HEADS, N_POSITIONS, ROPE_BASE)
        self.in_dropout = nn.Dropout(DROPOUT)
        self.blocks     = nn.ModuleList([
            TransformerBlock(D_MODEL, N_HEADS, D_FF_GATE, DROPOUT, N_LAYERS, self.rope)
            for _ in range(N_LAYERS)
        ])
        self.norm_final = RMSNorm(D_MODEL)
        self.out_to_k   = nn.Linear(D_MODEL, K_OUT, bias=False)

        # Warm-start: align out_to_k rows to B's subspace immediately.
        # For k_in != k_out, we copy the first k_out rows of B.T.
        with torch.no_grad():
            self.out_to_k.weight.copy_(self.embed.B.weight.T[:K_OUT, :])

    def forward_features(self, ids):
        h = self.in_dropout(self.embed.embed(ids))
        for blk in self.blocks:
            h = blk(h)
        return self.norm_final(h)
    def forward(self, ids, labels=None):
        h = self.forward_features(ids)                      # [B, T, D]

    # Project to k-space, then decode via vMF scoring
        z      = self.out_to_k(h).float()                  # [B, T, k]  unnormalized
        logits = self.embed.decode_scores(z)               # [B, T, V]  κ·cos(ẑ, Ã_out)

        loss = None
        if labels is not None:
            loss = F.cross_entropy(
              logits.view(-1, VOCAB_SIZE),
              labels.view(-1),
          )

    # Still expose z for κ diagnostics in training loop
        kappa = z.norm(dim=-1, keepdim=True)
        return SimpleNamespace(loss=loss, z=z, kappa=kappa, logits=logits)

    def count_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# ── E4 Diagnostic: cosine similarity between A_in and A_out ──────────────────
@torch.no_grad()
def log_embedding_alignment(model, step, epoch):
    """
    Core E4 falsification diagnostic.

    When k_in == k_out, we compute per-token cosine directly.
    When k_in != k_out, we compute cosine on the shared subspace:
      A_in[:, :k_shared] vs A_out[:, :k_shared], k_shared=min(k_in, k_out).

    This keeps the diagnostic shape-safe while preserving a comparable trend.
    Also logs ‖A_in‖ vs ‖A_out‖ to monitor norm drift.
    """
    A_in  = model.embed.A_in.weight.float()
    A_out = model.embed.A_out.weight.float()

    k_in, k_out = A_in.size(1), A_out.size(1)
    k_shared = min(k_in, k_out)

    A_in_cmp = A_in[:, :k_shared]
    A_out_cmp = A_out[:, :k_shared]

    A_in_n  = F.normalize(A_in_cmp,  dim=-1)
    A_out_n = F.normalize(A_out_cmp, dim=-1)

    # Per-token cosine similarity in shared subspace.
    cos_sim = (A_in_n * A_out_n).sum(dim=-1)   # [V]

    # norms
    norm_in  = A_in.norm(dim=-1)               # [V]
    norm_out = A_out.norm(dim=-1)              # [V]

    print(
        f'[E4 diag] ep{epoch} step{step:>5} | '
        f'cos_shared(A_in,A_out): mean={cos_sim.mean():.4f} std={cos_sim.std():.4f} '
        f'min={cos_sim.min():.4f} max={cos_sim.max():.4f} '
        f'(k_shared={k_shared}, k_in={k_in}, k_out={k_out}) | '
        f'||A_in||: mean={norm_in.mean():.3f} | '
        f'||A_out||: mean={norm_out.mean():.3f}'
    )
    return cos_sim.mean().item()


# ── Data loading (unchanged) ──────────────────────────────────────────────────
def _copy_if_needed(src, dst):
    if not src.exists():
        return False
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not dst.exists():
        print(f'Copying cache: {src} -> {dst}')
        shutil.copy2(src, dst)
    return True


def _tokenize_split(tokenizer, texts, split_name, batch_size=TOKENIZE_BATCH_SIZE):
    texts   = [t for t in texts if t and t.strip()]
    all_ids = []
    print(f'[{split_name}] tokenizing {len(texts):,} non-empty documents')
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc   = tokenizer(batch, add_special_tokens=False, truncation=False)
        for ids in enc['input_ids']:
            if ids:
                all_ids.extend(ids)
                all_ids.append(tokenizer.eos_token_id)
        if (i // batch_size) % 20 == 0:
            print(f'[{split_name}] {min(i + batch_size, len(texts)):,}/{len(texts):,} docs')
    tokens = torch.tensor(all_ids, dtype=torch.long)
    print(f'[{split_name}] tokens={tokens.numel():,}')
    return tokens


def load_wikitext103():
    tokenizer = GPT2TokenizerFast.from_pretrained(TOKENIZER_NAME)
    tokenizer.pad_token = tokenizer.eos_token
    LOCAL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    drive_files = {s: DRIVE_CACHE_DIR / f'{s}_tokens.pt' for s in ['train', 'val', 'test']}
    local_files = {s: LOCAL_CACHE_DIR  / f'{s}_tokens.pt' for s in ['train', 'val', 'test']}
    for split in ['train', 'val', 'test']:
        _copy_if_needed(drive_files[split], local_files[split])
    if all(local_files[s].exists() for s in ['train', 'val', 'test']):
        print('Loading cached token tensors from local cache...')
        train_tokens = torch.load(local_files['train'], map_location='cpu')
        val_tokens   = torch.load(local_files['val'],   map_location='cpu')
        test_tokens  = torch.load(local_files['test'],  map_location='cpu')
        print(f'train: {train_tokens.numel():,} | val: {val_tokens.numel():,} | test: {test_tokens.numel():,}')
        return tokenizer, train_tokens, val_tokens, test_tokens
    print('Cache missing — tokenizing WikiText-103...')
    raw          = load_dataset('wikitext', 'wikitext-103-raw-v1')
    train_tokens = _tokenize_split(tokenizer, raw['train']['text'],      'train')
    val_tokens   = _tokenize_split(tokenizer, raw['validation']['text'], 'val')
    test_tokens  = _tokenize_split(tokenizer, raw['test']['text'],       'test')
    for s, t in [('train', train_tokens), ('val', val_tokens), ('test', test_tokens)]:
        torch.save(t, local_files[s])
    try:
        DRIVE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
        for s in ['train', 'val', 'test']:
            shutil.copy2(local_files[s], drive_files[s])
    except Exception as e:
        print(f'Warning: could not copy to Drive: {e}')
    return tokenizer, train_tokens, val_tokens, test_tokens


def build_loaders(train_tokens, val_tokens):
    train_loader = DataLoader(
        TokenDataset(train_tokens, SEQ_LEN),
        batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
    )
    val_loader = DataLoader(
        TokenDataset(val_tokens, SEQ_LEN),
        batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True, drop_last=False,
    )
    return train_loader, val_loader


# ── Evaluation ────────────────────────────────────────────────────────────────
def evaluate(model, loader, max_batches=None, split_name='val'):
    model.eval()
    correct, total, loss_sum, steps = 0, 0, 0.0, 0
    kappa_mean_sum, kappa_min_sum, kappa_max_sum = 0.0, 0.0, 0.0
    with torch.no_grad():
        for step, batch in enumerate(loader):
            if max_batches is not None and step >= max_batches:
                break
            x, y  = [t.to(device, non_blocking=True) for t in batch]
            dtype = torch.bfloat16 if USE_BF16 else torch.float32
            with torch.amp.autocast('cuda', dtype=dtype, enabled=USE_BF16):
                out = model(x, labels=y)
            preds          = out.logits.argmax(dim=-1)
            correct       += (preds == y).sum().item()
            total         += y.numel()
            loss_sum      += out.loss.item()
            kappa_mean_sum += out.kappa.mean().item()
            kappa_min_sum  += out.kappa.min().item()
            kappa_max_sum  += out.kappa.max().item()
            steps += 1
    recall1  = correct   / max(total, 1)
    avg_loss = loss_sum  / max(steps, 1)
    ppl      = math.exp(min(avg_loss, 20))
    k_mean   = kappa_mean_sum / max(steps, 1)
    k_min    = kappa_min_sum  / max(steps, 1)
    k_max    = kappa_max_sum  / max(steps, 1)
    print(
        f'{split_name} recall@1={recall1:.6f} | ce={avg_loss:.6f} | ppl={ppl:.2f} | '
        f'κ mean={k_mean:.3f} min={k_min:.3f} max={k_max:.3f}'
    )
    return {'recall@1': recall1, 'ce_loss': avg_loss, 'ppl': ppl,
            'kappa_mean': k_mean, 'kappa_min': k_min, 'kappa_max': k_max}


def evaluate_ppl_sliding(model, tokens, stride=EVAL_STRIDE, seq_len=SEQ_LEN):
    model.eval()
    n = tokens.numel()
    nll_sum, ntok, prev_end = 0.0, 0, 0
    with torch.no_grad():
        for s in range(0, n - 1, stride):
            e  = min(s + seq_len, n - 1)
            x  = tokens[s:e].unsqueeze(0).to(device)
            y  = tokens[s + 1:e + 1].unsqueeze(0).to(device)
            pred_start = max(prev_end - s, 0) if s > 0 else 0
            if pred_start >= x.size(1):
                prev_end = e
                continue
            y_in = y.clone()
            if pred_start > 0:
                y_in[:, :pred_start] = -100
            dtype = torch.bfloat16 if USE_BF16 else torch.float32
            with torch.amp.autocast('cuda', dtype=dtype, enabled=USE_BF16):
                out = model(x, labels=y_in)
            valid     = (y_in != -100).sum().item()
            nll_sum  += out.loss.item() * valid
            ntok     += valid
            prev_end  = e
            if e >= n - 1:
                break
    return math.exp(min(nll_sum / max(ntok, 1), 20))


# ── Optimizer & scheduler ─────────────────────────────────────────────────────
def make_optimizer(model):
    decay, no_decay = [], []
    for _, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (decay if p.ndim >= 2 else no_decay).append(p)
    return torch.optim.AdamW([
        {'params': decay,    'weight_decay': 0.1},
        {'params': no_decay, 'weight_decay': 0.0},
    ], lr=LR, betas=(0.9, 0.95), eps=1e-8)


def make_scheduler(optimizer, steps_per_epoch):
    total_steps = SCHEDULE_HORIZON_EPOCHS * steps_per_epoch
    def lr_lambda(step):
        if step < WARMUP_STEPS:
            return float(step + 1) / float(max(1, WARMUP_STEPS))
        progress = (step - WARMUP_STEPS) / float(max(1, total_steps - WARMUP_STEPS))
        return 0.5 * (1.0 + math.cos(math.pi * min(max(progress, 0.0), 1.0)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


# ── Training loop ─────────────────────────────────────────────────────────────
def train_model(model, train_loader, val_loader, test_tokens):
    print(f'Training {RUN_NAME} | params={model.count_params():,}')
    print(
        'E4 gradient routing:\n'
        '  A_in.weight   ← g_in  only  (sparse, free to specialize as transformer input)\n'
        '  A_out.weight  ← g_out only  (dense,  free to specialize as retrieval geometry)\n'
        '  B.weight      ← g_in  only  (upscale, unchanged from E3)\n'
        '  out_to_k      ← g_out only  (projection, unchanged from E3)\n'
        '  Warm-start: out_to_k.weight <- B.weight.T[:k_out]  (row-aligned init for k_in!=k_out)\n'
    )

    # Log initial alignment — should be high because of warm-start (B.T alignment),
    # but A_in and A_out start from independent random inits → cos ≈ 0 initially
    log_embedding_alignment(model, 0, 0)

    optimizer = make_optimizer(model)
    scheduler = make_scheduler(optimizer, len(train_loader))

    for epoch in range(1, EPOCHS + 1):
        model.train()
        t0           = time.time()
        running_loss = 0.0
        optimizer.zero_grad(set_to_none=True)

        for step, batch in enumerate(train_loader, start=1):
            x, y  = [t.to(device, non_blocking=True) for t in batch]
            dtype = torch.bfloat16 if USE_BF16 else torch.float32
            with torch.amp.autocast('cuda', dtype=dtype, enabled=USE_BF16):
                out  = model(x, labels=y)
                loss = out.loss / ACCUM_STEPS
            loss.backward()
            running_loss += out.loss.item()

            if step % ACCUM_STEPS == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            if step % 200 == 0:
                kappa_mean = out.kappa.mean().item()
                kappa_std  = out.kappa.std().item()
                print(
                    f'epoch {epoch} step {step}/{len(train_loader)} '
                    f'ce_loss {running_loss / step:.6f} '
                    f'lr {scheduler.get_last_lr()[0]:.2e} '
                    f'κ̄={kappa_mean:.3f} κ_std={kappa_std:.3f}'
                )

            # Log A_in vs A_out alignment at key checkpoints
            # Step 200: should show cos ≈ 0 (random inits diverging or converging?)
            # Step 1000: has warm-start eliminated the subspace rotation phase?
            # Step 2000: is the split stabilizing?
            if step in (200, 1000, 2000, 4000, 8000):
                model.eval()
                log_embedding_alignment(model, step, epoch)
                model.train()

        val_metrics = evaluate(model, val_loader, max_batches=VAL_MAX_BATCHES, split_name='val')
        test_ppl    = evaluate_ppl_sliding(model, test_tokens)
        train_ce    = running_loss / len(train_loader)

        # End-of-epoch full diagnostic
        cos_mean = log_embedding_alignment(model, len(train_loader), epoch)

        print(
            f'\nEpoch {epoch}: '
            f'train_ce={train_ce:.4f} '
            f'val_recall@1={val_metrics["recall@1"]:.6f} '
            f'val_ppl={val_metrics["ppl"]:.2f} '
            f'test_ppl={test_ppl:.2f} '
            f'cos_shared(A_in,A_out)={cos_mean:.4f} '
            f'time={time.time() - t0:.1f}s\n'
        )

        # Falsification check
        if epoch == 1:
            if val_metrics['recall@1'] < 0.32:
                print('[FALSIFIED] Epoch 1 recall < 0.32: A-conflict was not the bottleneck.')
            elif val_metrics['recall@1'] >= 0.34:
                print('[CONFIRMED] Epoch 1 recall ≥ 0.34: A-conflict diagnosis correct.')
            if cos_mean > 0.8:
                print('[FALSIFIED] cos_shared(A_in,A_out) > 0.8: tying appears close in shared subspace.')
            elif cos_mean < 0.3:
                print('[CONFIRMED] cos_shared(A_in,A_out) < 0.3: matrices diverged in shared subspace.')

        torch.cuda.empty_cache()
        gc.collect()


# ── Entry point ───────────────────────────────────────────────────────────────
def main():
    print(f'Device: {device}')
    if torch.cuda.is_available():
        print(f'GPU: {torch.cuda.get_device_name(0)} | BF16: {USE_BF16}')
    print(f'Run: {RUN_NAME}\n')
    print(
        '[DIAGNOSIS] The disease in E3: A is still tied.\n'
        '  With k=24: V/k = 2,094 gradient conflict ratio.\n'
        '  Output path dominates A: κ≈17, dense 100% coverage, magnitude ~17.\n'
        '  Input  path starved on A: magnitude ~0.35, 16.3% coverage per step.\n'
        '  AdamW effective: A is 97.9% retrieval matrix, 2.1% representation.\n'
        '[FIX] Split A → A_in + A_out. No shared matrix anywhere.\n'
        '[PREDICT] epoch1 recall ≥ 0.34 | epoch2 recall ≥ 0.38 | epoch2 ppl ≤ 65\n'
    )
    _, train_tokens, val_tokens, test_tokens = load_wikitext103()
    train_loader, val_loader = build_loaders(train_tokens, val_tokens)
    model = BreakthroughMicroTransformerE4().to(device)
    print(f'Params: {model.count_params():,}  (E3 was 4,758,360; +{model.count_params()-4_758_360:,})\n')
    train_model(model, train_loader, val_loader, test_tokens)


if __name__ == '__main__':
    main()


Device: cuda
GPU: Tesla T4 | BF16: True
Run: v13_E4_kin96_kout64

[DIAGNOSIS] The disease in E3: A is still tied.
  With k=24: V/k = 2,094 gradient conflict ratio.
  Output path dominates A: κ≈17, dense 100% coverage, magnitude ~17.
  Input  path starved on A: magnitude ~0.35, 16.3% coverage per step.
  AdamW effective: A is 97.9% retrieval matrix, 2.1% representation.
[FIX] Split A → A_in + A_out. No shared matrix anywhere.
[PREDICT] epoch1 recall ≥ 0.34 | epoch2 recall ≥ 0.38 | epoch2 ppl ≤ 65

Loading cached token tensors from local cache...
train: 119,085,169 | val: 249,750 | test: 286,178
Params: 11,614,816  (E3 was 4,758,360; +6,856,456)

Training v13_E4_kin96_kout64 | params=11,614,816
E4 gradient routing:
  A_in.weight   ← g_in  only  (sparse, free to specialize as transformer input)
  A_out.weight  ← g_out only  (dense,  free to specialize as retrieval geometry)
  B.weight      ← g_in  only  (upscale, unchanged from E3)
  out_to_k      ← g_out only  (projection, unchanged from

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
from google.colab import runtime
runtime.unassign()